# Project name (FALTA)
------
group ect(FLATA)

----
# Exploratory Data Analysis (EDA) 
## Summary

(FALTA)

## Index

(FALTA)


## Imports

In [41]:
#imports 
#(create utils file for functions)
import pandas as pd
import numpy as np
# Resto de imports (visualization, etc)

In [42]:
df_AIAI_Customers_Complete = pd.read_csv('../../data/DM_AIAI_CustomerDB.csv')
df_AIAI_Customers = df_AIAI_Customers_Complete.copy()

df_AIAI_Flights_Complete = pd.read_csv('../../data/DM_AIAI_FlightsDB.csv')
df_AIAI_Flights = df_AIAI_Flights_Complete.copy()

## Metadata

DM_AIAI_CustomerDB.csv

- *Loyalty#* - Unique customer identifier for loyalty program members
- *First Name* - Customer’s first name
- *Last Name* - Customer’s last name
- *Customer Name* - Customer’s full name (concatenated)
- *Country* - Customer’s country of residence
- *Province or State* - Customer’s province or state
- *City* - Customer’s city of residence
- *Latitude* - Geographic latitude coordinate of customer location
- *Longitude* - Geographic longitude coordinate of customer location
- *Postal code* - Customer’s postal/ZIP code
- *Gender* - Customer’s gender
- *Education* - Customer’s highest education level (Bachelor, College, etc.)
- *Location Code* - Urban/Suburban/Rural classification of customer residence
- *Income* - Customer’s annual income
- *Marital Status* - Customer’s marital status (Married, Single, Divorced)
- *LoyaltyStatus* - Current tier status in loyalty program (Star > Nova > Aurora)
- *EnrollmentDateOpening* - Date when customer joined the loyalty program
- *CancellationDate* - Date when customer left the program
- *Customer Lifetime Value* - Total calculated monetary value of customer relationship
- *EnrollmentType* - Method of joining loyalty program

DM_AIAI_FlightsDB.csv


- *Loyalty#* - Unique customer identifier linking to CustomerDB
- *Year* - Year of flight activity record
- *Month* - Month of flight activity record (1-12)
- *YearMonthDate* - First day of the month for the activity period
- *NumFlights* -  Total number of flights taken by customer in the month
- *NumFlightsWithCompanions* - Number of flights where customer traveled with companions
- *DistanceKM* - Total distance traveled in kilometers for the month
- *PointsAccumulated* - Loyalty points earned by customer during the month
- *PointsRedeemed* - Loyalty points spent/redeemed by customer during the month
- *DollarCostPointsRedeemed* - Dollar value of points redeemed during the month


## Data Exploration

#### **Customers DataFrame**

In [43]:
df_AIAI_Customers.head()

,Unnamed: 0,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,...,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
0,0,480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,...,female,Bachelor,Urban,70146.0,Married,Star,2/15/2019,NaN,3839.14,Standard
1,1,549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,...,male,College,Rural,0.0,Divorced,Star,3/9/2019,NaN,3839.61,Standard
2,2,429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,...,male,College,Urban,0.0,Single,Star,7/14/2017,1/8/2021,3839.75,Standard
3,3,608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,...,male,College,Suburban,0.0,Single,Star,2/17/2016,NaN,3839.75,Standard
4,4,530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,...,male,Bachelor,Suburban,97832.0,Married,Star,10/25/2017,NaN,3842.79,2021 Promotion


Delete the first column that is not an index nor in the metadata

In [44]:
df_AIAI_Customers.drop(columns=['Unnamed: 0'], inplace=True)

Now we will verify the different types of data and the missing values

In [45]:
df_AIAI_Customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16921 entries, 0 to 16920
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Loyalty#                 16921 non-null  int64  
 1   First Name               16921 non-null  object 
 2   Last Name                16921 non-null  object 
 3   Customer Name            16921 non-null  object 
 4   Country                  16921 non-null  object 
 5   Province or State        16921 non-null  object 
 6   City                     16921 non-null  object 
 7   Latitude                 16921 non-null  float64
 8   Longitude                16921 non-null  float64
 9   Postal code              16921 non-null  object 
 10  Gender                   16921 non-null  object 
 11  Education                16921 non-null  object 
 12  Location Code            16921 non-null  object 
 13  Income                   16901 non-null  float64
 14  Marital Status        

We have on total 16921 observations.

The data types are correct on almost every column, meaning that the values stores at least are of the data types that they are supouse to be.

It is only needed to change the data types of the date columns (EnrollmentDateOpening, CancellationDate).

In [46]:
df_AIAI_Customers.isna().sum()

Loyalty#                       0
First Name                     0
Last Name                      0
Customer Name                  0
Country                        0
Province or State              0
City                           0
Latitude                       0
Longitude                      0
Postal code                    0
Gender                         0
Education                      0
Location Code                  0
Income                        20
Marital Status                 0
LoyaltyStatus                  0
EnrollmentDateOpening          0
CancellationDate           14611
Customer Lifetime Value       20
EnrollmentType                 0
dtype: int64

We have missing values in our data in the columns: Income (20 records, 0.11%), CancellationDate (14611 records, 86%), Customer Lifetime Value (20 records, 0.11%). We will evaluate and them decide if we can remove those observations (since they are a small proportion of the hole dataset) or do imputation.

Probably people do not fill the income because they do not want us to know how much they earn, so we cuold perform an imputation with similar data points.

The higher number of missing values in CancellationDate may be because not many people canceled and didn't leave the program. So it is not an error.

To know what to do with Customer Lifetime Value, we will need to see the relation that exits with the flights data, if the customer have some records if better not to remove and instead do some imputation based on similar points (performing imputation).

In [47]:
df_AIAI_Customers[df_AIAI_Customers[["Income", "Customer Lifetime Value"]].isna().sum(axis=1) > 1]

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
16901,999987,Layla,Murphy,Layla Murphy,Canada,New Brunswick,Fredericton,46.029263,-66.565150,R4H 2Y2,female,Bachelor,Urban,NaN,Single,Star,3/7/2017,3/7/2017,NaN,Standard
16902,999988,Jana,Parker,Jana Parker,Canada,Quebec,Montreal,45.573672,-73.523012,N6B 1N3,male,College,Rural,NaN,Single,Star,8/22/2017,8/22/2017,NaN,Standard
16903,999989,Ethan,Parker,Ethan Parker,Canada,Ontario,Trenton,44.075379,-77.550375,P8F 5C8,male,College,Rural,NaN,Married,Star,9/12/2015,9/12/2015,NaN,Standard
16904,999990,Ryan,Anderson,Ryan Anderson,Canada,New Brunswick,Moncton,46.106617,-64.714267,B6P 6D0,female,College,Rural,NaN,Married,Star,6/10/2019,6/10/2019,NaN,Standard
16905,999991,Olivia,Cote,Olivia Cote,Canada,New Brunswick,Fredericton,45.950000,-66.652437,X3W 5N2,female,College,Suburban,NaN,Married,Star,7/20/2019,7/20/2019,NaN,Standard
16906,999992,Ella,Roy,Ella Roy,Canada,Ontario,Toronto,43.706878,-79.437412,P6D 6N2,male,College,Suburban,NaN,Single,Star,3/27/2021,3/27/2021,NaN,Standard
16907,999993,Elijah,Cook,Elijah Cook,Canada,British Columbia,Dawson Creek,55.701475,-120.181716,W6H 0Z7,female,College,Suburban,NaN,Married,Star,1/27/2015,1/27/2015,NaN,Standard
16908,999994,Ethan,Chan,Ethan Chan,Canada,Ontario,Ottawa,45.365906,-75.723181,B2F 3E1,female,College,Rural,NaN,Married,Star,5/5/2016,5/5/2016,NaN,Standard
16909,999995,Liam,Wong,Liam Wong,Canada,Ontario,Ottawa,45.471557,-75.704868,B3A 2R0,female,College,Suburban,NaN,Married,Star,3/2/2020,3/2/2020,NaN,Standard
16910,999996,Isabella,Ross,Isabella Ross,Canada,Ontario,Toronto,43.690489,-79.436758,B4W 4M6,female,Bachelor,Suburban,NaN,Single,Star,9/14/2018,9/14/2018,NaN,Standard


In [48]:
df_AIAI_Customers[df_AIAI_Customers["EnrollmentDateOpening"] == df_AIAI_Customers["CancellationDate"] ]

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
3153,488724,Rayford,Vogus,Rayford Vogus,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,male,Bachelor,Suburban,36495.0,Married,Aurora,1/5/2016,1/5/2016,10629.22,Standard
12237,871455,Gemma,Gadbois,Gemma Gadbois,Canada,Alberta,Calgary,51.048615,-114.070850,T3E 2V9,male,Doctor,Suburban,16269.0,Divorced,Star,6/18/2020,6/18/2020,3211.07,Standard
16901,999987,Layla,Murphy,Layla Murphy,Canada,New Brunswick,Fredericton,46.029263,-66.565150,R4H 2Y2,female,Bachelor,Urban,NaN,Single,Star,3/7/2017,3/7/2017,NaN,Standard
16902,999988,Jana,Parker,Jana Parker,Canada,Quebec,Montreal,45.573672,-73.523012,N6B 1N3,male,College,Rural,NaN,Single,Star,8/22/2017,8/22/2017,NaN,Standard
16903,999989,Ethan,Parker,Ethan Parker,Canada,Ontario,Trenton,44.075379,-77.550375,P8F 5C8,male,College,Rural,NaN,Married,Star,9/12/2015,9/12/2015,NaN,Standard
16904,999990,Ryan,Anderson,Ryan Anderson,Canada,New Brunswick,Moncton,46.106617,-64.714267,B6P 6D0,female,College,Rural,NaN,Married,Star,6/10/2019,6/10/2019,NaN,Standard
16905,999991,Olivia,Cote,Olivia Cote,Canada,New Brunswick,Fredericton,45.950000,-66.652437,X3W 5N2,female,College,Suburban,NaN,Married,Star,7/20/2019,7/20/2019,NaN,Standard
16906,999992,Ella,Roy,Ella Roy,Canada,Ontario,Toronto,43.706878,-79.437412,P6D 6N2,male,College,Suburban,NaN,Single,Star,3/27/2021,3/27/2021,NaN,Standard
16907,999993,Elijah,Cook,Elijah Cook,Canada,British Columbia,Dawson Creek,55.701475,-120.181716,W6H 0Z7,female,College,Suburban,NaN,Married,Star,1/27/2015,1/27/2015,NaN,Standard
16908,999994,Ethan,Chan,Ethan Chan,Canada,Ontario,Ottawa,45.365906,-75.723181,B2F 3E1,female,College,Rural,NaN,Married,Star,5/5/2016,5/5/2016,NaN,Standard


In [49]:
df_AIAI_Customers[df_AIAI_Customers[["Income", "Customer Lifetime Value"]].isna().sum(axis=1) > 1].info()

<class 'pandas.core.frame.DataFrame'>
Index: 20 entries, 16901 to 16920
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Loyalty#                 20 non-null     int64  
 1   First Name               20 non-null     object 
 2   Last Name                20 non-null     object 
 3   Customer Name            20 non-null     object 
 4   Country                  20 non-null     object 
 5   Province or State        20 non-null     object 
 6   City                     20 non-null     object 
 7   Latitude                 20 non-null     float64
 8   Longitude                20 non-null     float64
 9   Postal code              20 non-null     object 
 10  Gender                   20 non-null     object 
 11  Education                20 non-null     object 
 12  Location Code            20 non-null     object 
 13  Income                   0 non-null      float64
 14  Marital Status           2

When the customer has the missing on the income variable, it also do not have Lifetime Value, and they canceled the suscription on the same day, meaning these are persons that are not really customers. At this point the aproach of remove those observations is the better.

Is important to say that are 2 customers that enrolled and canceled on the same day but they have some monetary value, meaning that they only purchased on 1 day, but they are customers (loyalty#: 488724 and 871455).

In [50]:
df_AIAI_Flights[df_AIAI_Flights["Loyalty#"].isin([488724,871455])].describe(include="all")

,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
count,72.000000,72.000000,72.000000,72,72.0,72.0,72.0,72.0,72.0,72.0
unique,NaN,NaN,NaN,36,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,6/1/2020,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN
mean,680089.500000,2020.000000,6.500000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
std,192708.432265,0.822226,3.476278,NaN,0.0,0.0,0.0,0.0,0.0,0.0
min,488724.000000,2019.000000,1.000000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
25%,488724.000000,2019.000000,3.750000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
50%,680089.500000,2020.000000,6.500000,NaN,0.0,0.0,0.0,0.0,0.0,0.0
75%,871455.000000,2021.000000,9.250000,NaN,0.0,0.0,0.0,0.0,0.0,0.0


We will also drop these two customers since they dont have any real records of flights (no variance on their transactions)

**Duplicates**

In [51]:
df_AIAI_Customers.duplicated().sum()
# No duplicate rows found

np.int64(0)

In [52]:
df_AIAI_Customers.duplicated(subset=['Loyalty#'], keep=False).sum()
# 327 duplicate Loyalty# values found

np.int64(327)

In [53]:
duplicates_loyalty_Customers_mask = df_AIAI_Customers.duplicated(subset=['Loyalty#'], keep=False)
df_AIAI_Customers[duplicates_loyalty_Customers_mask].sort_values(['Loyalty#'])

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
1646,101902,Hans,Schlottmann,Hans Schlottmann,Canada,Ontario,London,42.984924,-81.245277,M5B 3E4,female,College,Rural,0.0,Married,Aurora,1/7/2020,NaN,6265.34,Standard
2668,101902,Yi,Nesti,Yi Nesti,Canada,Ontario,Toronto,43.653225,-79.383186,M8Y 4K8,female,Bachelor,Urban,79090.0,Married,Aurora,3/19/2020,NaN,8609.16,Standard
15988,106001,Maudie,Hyland,Maudie Hyland,Canada,New Brunswick,Fredericton,45.963589,-66.643112,E3B 2H2,female,Master,Suburban,14973.0,Divorced,Star,7/16/2015,NaN,12168.74,Standard
700,106001,Ivette,Peifer,Ivette Peifer,Canada,Quebec,Montreal,45.501690,-73.567253,H2Y 4R4,female,High School or Below,Suburban,10037.0,Single,Star,1/11/2016,NaN,4914.04,Standard
13053,106509,Stacy,Schwebke,Stacy Schwebke,Canada,Ontario,Toronto,43.653225,-79.383186,P1J 8T7,female,College,Suburban,0.0,Single,Star,6/12/2021,NaN,4661.98,Standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5038,989528,Sharri,Boughman,Sharri Boughman,Canada,Quebec,Montreal,45.501690,-73.567253,H2T 2J6,female,College,Rural,0.0,Divorced,Nova,5/1/2020,NaN,3370.07,Standard
9890,990512,Magda,Sopher,Magda Sopher,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,10/21/2018,NaN,1904.00,Standard
14478,990512,Ione,Snowden,Ione Snowden,Canada,British Columbia,Vancouver,49.282730,-123.120740,V5R 1W3,female,College,Urban,0.0,Single,Star,8/20/2021,NaN,6870.61,Standard
16380,992168,Crysta,Bennin,Crysta Bennin,Canada,Ontario,Ottawa,45.421532,-75.697189,K1F 2R2,female,Master,Urban,22828.0,Married,Star,12/18/2017,NaN,16473.17,Standard


We will drop these records since there are people with the same Loyalty number and different characteristics, and we do not know which is the "real person", and since we need to use that loyalty number to identify the customer in the other dataset we will drop them (are only 327 rows out of 16921, so it is only 1.93% of the dataset).

And we can put as index colunm the Loyalty number since now we know that will be unique.

In [54]:
df_AIAI_Customers[df_AIAI_Customers['CancellationDate'] == "2/29/2019"]
# We found that there is a wrong date format, since that day doesnt exist, we will change it to 2/28/2019

,Loyalty#,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
8840,314558,Retta,Pauley,Retta Pauley,Canada,Quebec,Montreal,45.501690,-73.567253,H4G 3T4,male,Bachelor,Rural,54311.0,Married,Nova,9/5/2017,2/29/2019,10722.06,Standard
14757,373118,Quinn,Shamonsky,Quinn Shamonsky,Canada,Manitoba,Winnipeg,49.895138,-97.138374,R3R 3T4,male,College,Rural,0.0,Single,Star,6/29/2018,2/29/2019,7516.79,Standard


In [55]:
df_AIAI_Customers.loc[df_AIAI_Customers['CancellationDate'] == "2/29/2019", 'CancellationDate'] = "2/28/2019"

In [56]:
# Removing duplicate Loyalty# rows
df_AIAI_Customers = df_AIAI_Customers[~(duplicates_loyalty_Customers_mask)]

# Removing rows with missing Income and Customer Lifetime Value
df_AIAI_Customers = df_AIAI_Customers[~(df_AIAI_Customers[["Income", "Customer Lifetime Value"]].isna().sum(axis=1) > 1)]

# Removing the two customers that enrolled and canceled on the same day
df_AIAI_Customers = df_AIAI_Customers[~(df_AIAI_Customers["Loyalty#"].isin([488724,871455]))]

# Setting Loyalty# as index
df_AIAI_Customers.set_index('Loyalty#', inplace=True)

# Putting the date columns on datetime format
df_AIAI_Customers['EnrollmentDateOpening'] = pd.to_datetime(df_AIAI_Customers['EnrollmentDateOpening'], format='%m/%d/%Y')
df_AIAI_Customers['CancellationDate'] = pd.to_datetime(df_AIAI_Customers['CancellationDate'], format='%m/%d/%Y')

In [57]:
df_AIAI_Customers.head()

,First Name,Last Name,Customer Name,Country,Province or State,City,Latitude,Longitude,Postal code,Gender,Education,Location Code,Income,Marital Status,LoyaltyStatus,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value,EnrollmentType
Loyalty#,,,,,,,,,,,,,,,,,,,
480934,Cecilia,Householder,Cecilia Householder,Canada,Ontario,Toronto,43.653225,-79.383186,M2Z 4K1,female,Bachelor,Urban,70146.0,Married,Star,2019-02-15,NaT,3839.14,Standard
549612,Dayle,Menez,Dayle Menez,Canada,Alberta,Edmonton,53.544388,-113.490930,T3G 6Y6,male,College,Rural,0.0,Divorced,Star,2019-03-09,NaT,3839.61,Standard
429460,Necole,Hannon,Necole Hannon,Canada,British Columbia,Vancouver,49.282730,-123.120740,V6E 3D9,male,College,Urban,0.0,Single,Star,2017-07-14,2021-01-08,3839.75,Standard
608370,Queen,Hagee,Queen Hagee,Canada,Ontario,Toronto,43.653225,-79.383186,P1W 1K4,male,College,Suburban,0.0,Single,Star,2016-02-17,NaT,3839.75,Standard
530508,Claire,Latting,Claire Latting,Canada,Quebec,Hull,45.428730,-75.713364,J8Y 3Z5,male,Bachelor,Suburban,97832.0,Married,Star,2017-10-25,NaT,3842.79,2021 Promotion


In [58]:
df_AIAI_Customers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16572 entries, 480934 to 652627
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   First Name               16572 non-null  object        
 1   Last Name                16572 non-null  object        
 2   Customer Name            16572 non-null  object        
 3   Country                  16572 non-null  object        
 4   Province or State        16572 non-null  object        
 5   City                     16572 non-null  object        
 6   Latitude                 16572 non-null  float64       
 7   Longitude                16572 non-null  float64       
 8   Postal code              16572 non-null  object        
 9   Gender                   16572 non-null  object        
 10  Education                16572 non-null  object        
 11  Location Code            16572 non-null  object        
 12  Income                   16572 

In [59]:
df_AIAI_Customers.isna().sum()

First Name                     0
Last Name                      0
Customer Name                  0
Country                        0
Province or State              0
City                           0
Latitude                       0
Longitude                      0
Postal code                    0
Gender                         0
Education                      0
Location Code                  0
Income                         0
Marital Status                 0
LoyaltyStatus                  0
EnrollmentDateOpening          0
CancellationDate           14327
Customer Lifetime Value        0
EnrollmentType                 0
dtype: int64

(FALTA) % de valores eliminados, % de dropout%, chequekar que el enrolement sea mas nuevo que el cancel

In [60]:
df_AIAI_Customers.describe()

,Latitude,Longitude,Income,EnrollmentDateOpening,CancellationDate,Customer Lifetime Value
count,16572.000000,16572.000000,16572.000000,16572,2245,16572.000000
mean,47.174612,-91.839905,37741.357531,2018-10-07 04:29:43.055756800,2019-12-20 11:55:49.844098048,7986.623409
min,42.984924,-135.056840,0.000000,2015-04-01 00:00:00,2015-11-30 00:00:00,1898.010000
25%,44.231171,-120.237660,0.000000,2017-01-18 18:00:00,2019-02-02 00:00:00,3979.127500
50%,46.087818,-79.383186,34137.000000,2018-11-02 00:00:00,2020-01-14 00:00:00,5780.180000
75%,49.282730,-74.596184,62375.000000,2020-07-11 06:00:00,2021-02-15 00:00:00,8954.430000
max,60.721188,-52.712578,99981.000000,2021-12-30 00:00:00,2021-12-30 00:00:00,83325.380000
std,3.305572,22.240873,30357.279626,NaN,NaN,6858.782006


In [61]:
df_AIAI_Customers.describe(include="O")

,First Name,Last Name,Customer Name,Country,Province or State,City,Postal code,Gender,Education,Location Code,Marital Status,LoyaltyStatus,EnrollmentType
count,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572,16572
unique,4926,15114,16572,1,11,29,55,2,5,3,3,3,2
top,Stacey,Salberg,Ariane Peyton,Canada,Ontario,Toronto,V6E 3D9,female,Bachelor,Suburban,Married,Star,Standard
freq,13,4,1,16572,5353,3322,906,8335,10377,5606,9645,7597,15434


(Falata) Toda la interpretacion de las stats y eliminar la variable country que son tdos de canada.
Analisar los codigos postales porque son muy pocos en comparacion a las personas y ver si la longitud y latitud son la misma, sino hay un error en los codigos postales

(ME QUEDE AQUI)

#### **Flights DataFrame**

In [93]:
df_AIAI_Flights.head()

,Loyalty#,Year,Month,YearMonthDate,NumFlights,NumFlightsWithCompanions,DistanceKM,PointsAccumulated,PointsRedeemed,DollarCostPointsRedeemed
0,413052,2021,12,12/1/2021,2.0,2.0,9384.0,938.0,0.0,0.0
1,464105,2021,12,12/1/2021,0.0,0.0,0.0,0.0,0.0,0.0
2,681785,2021,12,12/1/2021,10.0,3.0,14745.0,1474.0,0.0,0.0
3,185013,2021,12,12/1/2021,16.0,4.0,26311.0,2631.0,3213.0,32.0
4,216596,2021,12,12/1/2021,9.0,0.0,19275.0,1927.0,0.0,0.0


In [94]:
df_AIAI_Flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608436 entries, 0 to 608435
Data columns (total 10 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Loyalty#                  608436 non-null  int64  
 1   Year                      608436 non-null  int64  
 2   Month                     608436 non-null  int64  
 3   YearMonthDate             608436 non-null  object 
 4   NumFlights                608436 non-null  float64
 5   NumFlightsWithCompanions  608436 non-null  float64
 6   DistanceKM                608436 non-null  float64
 7   PointsAccumulated         608436 non-null  float64
 8   PointsRedeemed            608436 non-null  float64
 9   DollarCostPointsRedeemed  608436 non-null  float64
dtypes: float64(6), int64(3), object(1)
memory usage: 46.4+ MB


In [15]:
df_AIAI_Flights.isna().sum()

Loyalty#                    0
Year                        0
Month                       0
YearMonthDate               0
NumFlights                  0
NumFlightsWithCompanions    0
DistanceKM                  0
PointsAccumulated           0
PointsRedeemed              0
DollarCostPointsRedeemed    0
dtype: int64

There are no missing values

In [51]:
# Value Count completar

In [9]:
df_AIAI_Flights.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Loyalty#,608436.0,NaN,NaN,NaN,550037.873084,258935.180575,100018.0,326961.0,550834.0,772194.0,999986.0
Year,608436.0,NaN,NaN,NaN,2020.0,0.816497,2019.0,2019.0,2020.0,2021.0,2021.0
Month,608436.0,NaN,NaN,NaN,6.5,3.452055,1.0,3.75,6.5,9.25,12.0
YearMonthDate,608436,36,12/1/2021,16901,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NumFlights,608436.0,NaN,NaN,NaN,3.908107,5.057889,0.0,0.0,0.0,7.2,21.0
NumFlightsWithCompanions,608436.0,NaN,NaN,NaN,0.983944,2.003785,0.0,0.0,0.0,0.9,11.0
DistanceKM,608436.0,NaN,NaN,NaN,7939.341419,10260.421873,0.0,0.0,856.4,15338.175,42040.0
PointsAccumulated,608436.0,NaN,NaN,NaN,793.777781,1025.918521,0.0,0.0,85.275,1533.7125,4204.0
PointsRedeemed,608436.0,NaN,NaN,NaN,235.251678,983.233374,0.0,0.0,0.0,0.0,7496.0
DollarCostPointsRedeemed,608436.0,NaN,NaN,NaN,2.324835,9.725168,0.0,0.0,0.0,0.0,74.0


In [43]:
df_AIAI_Flights.describe(include="O").T

,count,unique,top,freq
YearMonthDate,608436,36,12/1/2021,16901


# Duplicates

In [20]:
df_AIAI_Customers.duplicated().sum()

np.int64(0)

In [21]:
df_AIAI_Flights.duplicated().sum()

np.int64(2903)

In [49]:
# The unique identifier column is 'Loyalty#'

# 1. Get the unique IDs from each DataFrame
customer_ids = set(df_AIAI_Customers['Loyalty#'])
flight_ids = set(df_AIAI_Flights['Loyalty#'])

# 2. Find the IDs present in Flights but MISSING in Customers
# We use flight_ids - customer_ids to find the difference
missing_flight_ids = flight_ids - customer_ids
# Alternatively: orphan_flight_ids = flight_ids.difference(customer_ids)

# 3. Print the results
if len(missing_flight_ids) == 0:
    print("All Flight IDs are associated with a customer in the Customers dataset.")
else:
    print(f"Warning {len(missing_flight_ids)} Flight IDs are not found in the Customers dataset.")
    print("\nFirst 10 'Orphan' Flight IDs:")
    print(list(missing_flight_ids)[:10])


All Flight IDs are associated with a customer in the Customers dataset.


In [ ]:
# Perform the Full Outer Join
df_merged_full = pd.merge(
    left=df_AIAI_Customers,
    right=df_AIAI_Flights,
    on='Loyalty#', # The common key column
    how='outer',  # This ensures all rows from both datasets are kept
    indicator=True # (Optional) Adds a column to show where the data came from
)


print(df_merged_full)


# Value in _merge	Meaning
#   both	        The row has a match in both the Customers and Flights datasets.
#  left_only	    The row exists only in the Customers dataset (the flight columns will be NaN).
#  right_only	    The row exists only in the Flights dataset (the customer columns will be NaN).

In [ ]:
#DataFrame	           Column Name	         Current Type    Target Type
#f_AIAI_Customers	EnrollmentDateOpening	 object 	      datetime64
#df_AIAI_Customers	CancellationDate	     object           datetime64
#df_AIAI_Flights	YearMonthDate	         object 	      datetime64